# Example 5 (Python half): Scanning the Full SARS-CoV-2 Genome for ORF Lengths


Fetches the complete SARS-CoV-2 reference genome (`NC_045512.2`, 29,903 bp) and runs the
same six-frame scan as Examples 2 and 4, this time with a lower threshold (≥25 aa) so even
short, spurious ORFs are captured. Every ORF's length is saved to
`results/example5_orf_lengths.csv`, split by strand, for the R half to test.


SARS-CoV-2 is a positive-sense ssRNA virus — its genome *is* the mRNA, and
essentially every real gene (ORF1ab, S, E, M, N, accessory ORFs) sits on the strand
deposited as `NC_045512.2`.

In [2]:
# Fetch the full SARS-CoV-2 genome and save every ORF's length, per strand
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"
handle = Entrez.efetch(db="nucleotide", id="NC_045512.2", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()
SeqIO.write(record, "data/NC_045512.2_full_genome.fasta", "fasta")
print(f"Genome length: {len(record.seq)} bp")

def find_orf_lengths(seq, min_aa=25):
    """Same scanning logic as Examples 2 and 4 — this time collecting every length."""
    lengths = []
    for frame in range(3):
        trans = str(seq[frame:].translate(table=1, to_stop=False))
        start = None
        for i, aa in enumerate(trans):
            if aa == "M" and start is None:
                start = i
            if aa == "*" and start is not None:
                length = i - start
                if length >= min_aa:
                    lengths.append(length)
                start = None
    return lengths

forward_lengths = find_orf_lengths(record.seq, min_aa=25)
revcomp = record.seq.reverse_complement()
reverse_lengths = find_orf_lengths(revcomp, min_aa=25)

print(f"Forward-strand ORFs (>= 25 aa): {len(forward_lengths)}")
print(f"Reverse-strand ORFs (>= 25 aa): {len(reverse_lengths)}")
print(f"Longest forward ORF: {max(forward_lengths)} aa")
print(f"Longest reverse ORF: {max(reverse_lengths)} aa")

with open("results/example5_orf_lengths.csv", "w") as f:
    f.write("strand,length_aa\n")
    for length in forward_lengths:
        f.write(f"forward,{length}\n")
    for length in reverse_lengths:
        f.write(f"reverse,{length}\n")
print("\nSaved: data/NC_045512.2_full_genome.fasta, results/example5_orf_lengths.csv")

Genome length: 29903 bp
Forward-strand ORFs (>= 25 aa): 79
Reverse-strand ORFs (>= 25 aa): 80
Longest forward ORF: 4405 aa
Longest reverse ORF: 100 aa

Saved: data/NC_045512.2_full_genome.fasta, results/example5_orf_lengths.csv
